# 📘 Notebook 01 — Quickstart: From Zero to Agent-Ready

**What you'll learn:**
- How ctxtual stores tool results in workspaces instead of dumping them into the LLM context
- How to define producers, attach toolsets, and dispatch tool calls
- What the LLM actually sees (WorkspaceRef with JSON Schema, system prompt, tool schemas)

**Prerequisites:** `pip install ctxtual`

> 💡 No API keys needed — everything runs locally with simulated data.

## 1. The Problem: Tool Results That Don't Fit

In [ ]:
# Imagine your agent calls a search tool that returns 5,000 results.
# Without ctxtual, you'd stuff all of this into the LLM's context window:

fake_results = [
    {"id": i, "title": f"Paper #{i}", "year": 2020 + (i % 5), "citations": i * 3}
    for i in range(5000)
]

# This is ~2MB of JSON. Most LLMs cap at 128K tokens (~400KB).
import json
print(f"Raw payload size: {len(json.dumps(fake_results)):,} chars")
print(f"That's ~{len(json.dumps(fake_results)) // 4:,} tokens — way over budget.")

## 2. The Fix: Store It, Give the Agent a Map

In [ ]:
from ctxtual import Ctx, MemoryStore
from ctxtual.utils import paginator, text_search, filter_set, pipeline

# Create the ctx — the central orchestrator
ctx = Ctx(store=MemoryStore())

# Attach built-in toolsets for list-type data
pager   = paginator(ctx, "papers")
search  = text_search(ctx, "papers", fields=["title"])
filters = filter_set(ctx, "papers")
pipe    = pipeline(ctx, "papers")

## 3. Define a Producer

A producer is any function that returns data. The `@ctx.producer` decorator
intercepts the return value, stores it in a workspace, and returns a
**WorkspaceRef** — a compact notification the LLM can actually read.

In [ ]:
@ctx.producer(workspace_type="papers", toolsets=[pager, search, filters, pipe])
def search_papers(query: str, limit: int = 5000) -> list[dict]:
    """Search academic papers by keyword."""
    # In production, this would call an API or database.
    # For this demo, we generate synthetic data.
    return [
        {
            "id": i,
            "title": f"Paper about {query} — variant {i}",
            "author": ["Alice", "Bob", "Carol", "Dave"][i % 4],
            "year": 2019 + (i % 6),
            "citations": (i * 7) % 500,
            "abstract": f"This paper investigates {query} using method {i % 10}.",
        }
        for i in range(limit)
    ]

## 4. Call the Producer — See What the LLM Gets

In [ ]:
ref = search_papers("transformer architectures")

# This is what goes into the LLM's context — NOT 5,000 papers:
print(json.dumps(ref, indent=2))

👆 Notice:
- `item_count: 5000` — the LLM knows the dataset size
- `data_shape: "list"` — the LLM knows it's a list of dicts
- `item_schema` — a **JSON Schema** with field names AND types (integer, string, etc.)
- `available_tools` — the LLM knows *exactly* which tools to call next
- `next_steps` — actionable guidance for the LLM

The LLM knows `year` is an integer (so `$gte`/`$lte` filters work), `title` is a string
(so `$contains`/`$startswith` work). No need to paginate first to discover the structure.

**Total context cost: ~500 tokens instead of ~500,000.**

## 5. Dispatch Tool Calls (What the Agent Loop Does)

In [ ]:
workspace_id = ref["workspace_id"]

# The LLM would generate these tool calls. We simulate them here.

# 5a. Paginate — get page 0 (first 5 items)
page = ctx.dispatch_tool_call("papers_paginate", {
    "workspace_id": workspace_id,
    "page": 0,
    "size": 5,
})
# paginate wraps in {"result": ..., "_hint": "..."} envelope
data = page["result"]
print("=== Page 0 (5 items) ===")
for item in data["items"]:
    print(f"  [{item['id']}] {item['title']} ({item['year']}, {item['citations']} cites)")
print(f"  ... {data['total']} total items, {data['total_pages']} pages")

In [ ]:
# 5b. Search — find papers about "method 7"
results = ctx.dispatch_tool_call("papers_search", {
    "workspace_id": workspace_id,
    "query": "method 7",
    "max_results": 3,
})
# search returns raw dict (no envelope)
print("=== Search: 'method 7' ===")
for item in results["matches"]:
    print(f"  [{item['id']}] {item['title']}")
print(f"  {results['total_matches']} total matches")

In [ ]:
# 5c. Filter — papers from 2024
filtered = ctx.dispatch_tool_call("papers_filter_by", {
    "workspace_id": workspace_id,
    "field": "year",
    "value": 2024,
    "operator": "eq",
})
print(f"=== Filter: year == 2024 ===")
print(f"  {filtered['count']} papers from 2024")

In [ ]:
# 5d. Pipeline — compound query in ONE tool call
# "Top 5 most-cited papers from 2023+ by Alice, show title and citations"
result = ctx.dispatch_tool_call("papers_pipe", {
    "workspace_id": workspace_id,
    "steps": [
        {"filter": {"year": {"$gte": 2023}, "author": "Alice"}},
        {"sort": {"field": "citations", "order": "desc"}},
        {"limit": 5},
        {"select": ["title", "author", "year", "citations"]},
    ],
})
print("=== Pipeline: Top 5 Alice papers from 2023+ by citations ===")
for item in result["items"]:
    print(f"  {item['title']} — {item['citations']} cites ({item['year']})")

## 6. The System Prompt — Auto-Generated LLM Instructions

In [ ]:
print(ctx.system_prompt())

## 7. Tool Schemas for Function Calling

Export OpenAI-compatible schemas to pass to any LLM that supports function calling.

In [ ]:
schemas = ctx.get_all_tool_schemas()
print(f"{len(schemas)} tools registered:\n")
for s in schemas[:3]:  # Show first 3
    print(f"  📦 {s['function']['name']}")
    print(f"     {s['function']['description'][:80]}...")
    print(f"     params: {list(s['function']['parameters']['properties'].keys())}")
    print()
print(f"  ... and {len(schemas) - 3} more")

## 8. Error Recovery — What Happens When the LLM Makes Mistakes

In [ ]:
# Wrong workspace ID
result = ctx.dispatch_tool_call("papers_paginate", {
    "workspace_id": "nonexistent_abc123",
})
print("Wrong workspace_id:")
print(json.dumps(result, indent=2))

In [ ]:
# Wrong tool name
result = ctx.dispatch_tool_call("papers_nonexistent", {
    "workspace_id": workspace_id,
})
print("Wrong tool name:")
print(json.dumps(result, indent=2))

## Summary

| Concept | What It Does |
|---------|-------------|
| `Ctx` | Central orchestrator — manages store, toolsets, dispatch |
| `@ctx.producer` | Decorator that stores return values in workspaces |
| `WorkspaceRef` | Compact notification: count, shape, JSON Schema, tools |
| `dispatch_tool_call` | Routes tool name + args to the right function |
| `system_prompt()` | Auto-generated instructions for the LLM |
| `get_all_tool_schemas()` | OpenAI-format function calling schemas |

**Next:** [02_text_and_documents.ipynb](02_text_and_documents.ipynb) — handling raw text, webpages, PDFs